In [ ]:
# Notebook 1/8 (Database Generation) — Retracted OA PDF Downloader (Unpaywall)
#
# Purpose: This notebook downloads Open Access (OA) PDF files for the collection of
# retracted scientific articles using the Unpaywall API.
#
# The workflow implemented in this notebook is as follows:
#
# 1. Read a CSV file stored in Google Drive containing, at minimum:
#    - a unique case identifier ("code")
#    - a DOI ("doi")
#
# 2. Query the Unpaywall API for each DOI to determine whether an OA PDF
#    is available and retrieve the corresponding download URL.
#
# 3. Download the OA PDF when available and store it in a structured
#    Google Drive folder, naming each file using the case identifier.
#
# 4. Maintain a persistent CSV log recording:
#    - all attempted downloads
#    - success and failure states
#    - error messages when applicable
#
# 5. Maintain a small state file to allow safe resumption after
#    interruptions (e.g. Colab session timeouts).
#
# Inputs:
# - CSV file with columns:
#   * code : unique identifier used for file naming
#   * doi  : DOI to be queried via Unpaywall
#
# Outputs:
# - A folder of downloaded OA PDF files in Google Drive
# - A CSV log file with detailed download outcomes
# - A JSON state file supporting resumable execution
#
# Execution:
# - Run the notebook from top to bottom.
# - If execution is interrupted, rerunning the notebook will resume
#   automatically without duplicating completed work.
#
# Notes:
# This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pandas requests tqdm tenacity

In [ ]:
# 2) Imports

import os, re, time, json, requests
import pandas as pd
from tqdm import tqdm
from typing import Optional, Tuple
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

In [ ]:
# 3) Configuration

# Path to the CSV
INPUT_CSV = "//content/drive/MyDrive/Dissertation/retracted_dois.csv"

# Output folder
OUTPUT_DIR = "/content/drive/MyDrive/Dissertation/oa_pdfs"

# User email (required by Unpaywall)
UNPAYWALL_EMAIL = "xxxxxxxxxxxxx"   # required by Unpaywall

# Target number of successfully downloaded OA PDFs
TARGET_SUCCESSES = 5000

# Polite delay between iterations (helps avoid being rate-limited)
MIN_DELAY_S = 0.5

# Keep logs/ state in the same Drive folder (persistent + resumable)
LOG_PATH = os.path.join(OUTPUT_DIR, "/content/drive/MyDrive/Dissertation/download_log.csv")
STATE_PATH = os.path.join(OUTPUT_DIR, "/content/drive/MyDrive/Dissertation/state.json")

# Optional: if a file already exists and is bigger than this, count as already downloaded
MIN_VALID_FILESIZE_BYTES = 10_000

In [ ]:
# 4) Constants

UNPAYWALL_ENDPOINT = "https://api.unpaywall.org/v2/"

In [ ]:
# 5) Helpers

def sanitize_filename(s: str) -> str:
    """Make a safe filename from the code value."""
    s = str(s).strip()
    s = re.sub(r"[^\w\-. ]+", "_", s)     # replace unsafe chars
    s = re.sub(r"\s+", "_", s)            # spaces -> underscores
    return s[:180]                        # limit length

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase + strip columns for robust matching."""
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type(requests.RequestException),
)
def unpaywall_lookup(doi: str, email: str, timeout: int = 30) -> dict:
    """Query Unpaywall for OA metadata."""
    url = f"{UNPAYWALL_ENDPOINT}{doi}"
    resp = requests.get(url, params={"email": email}, timeout=timeout)

    # 404 means Unpaywall doesn't have the DOI in its index
    if resp.status_code == 404:
        return {}

    resp.raise_for_status()
    return resp.json()

def extract_oa_pdf_url(unpaywall_json: dict) -> Optional[str]:
    """Prefer best_oa_location.url_for_pdf; otherwise fall back to any oa_locations[].url_for_pdf."""
    if not unpaywall_json:
        return None

    best = unpaywall_json.get("best_oa_location") or {}
    url = best.get("url_for_pdf")
    if url:
        return url

    for loc in (unpaywall_json.get("oa_locations") or []):
        url = (loc or {}).get("url_for_pdf")
        if url:
            return url

    return None

@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=30),
    retry=retry_if_exception_type(requests.RequestException),
)
def download_pdf(pdf_url: str, out_path: str, timeout: int = 90) -> Tuple[bool, str]:
    """
    Download a PDF streaming to disk.
    Writes to .part file then moves into place (atomic-ish).
    Validates basic PDF signature.
    """
    headers = {
        "User-Agent": f"OA-PDF-Downloader/1.0 (mailto:{UNPAYWALL_EMAIL})"
    }

    with requests.get(pdf_url, stream=True, headers=headers, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()

        tmp_path = out_path + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

        # Validate PDF signature
        with open(tmp_path, "rb") as f:
            sig = f.read(5)
        if sig != b"%PDF-":
            try:
                os.remove(tmp_path)
            except Exception:
                pass
            return (False, "Downloaded content is not a valid PDF (missing %PDF- header).")

        os.replace(tmp_path, out_path)
        return (True, "OK")

def ensure_dirs():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def init_log():
    if not os.path.exists(LOG_PATH):
        cols = ["index", "code", "doi", "is_oa", "pdf_url", "status", "message", "filepath"]
        pd.DataFrame(columns=cols).to_csv(LOG_PATH, index=False)

def load_state() -> Tuple[int, int]:
    """Return (successes, last_index)."""
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "r", encoding="utf-8") as f:
            state = json.load(f)
        return int(state.get("successes", 0)), int(state.get("last_index", 0))
    return 0, 0

def save_state(successes: int, next_index: int):
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"successes": successes, "last_index": next_index}, f)

def append_log(row: dict):
    pd.DataFrame([row]).to_csv(LOG_PATH, mode="a", header=False, index=False)

def already_downloaded(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) >= MIN_VALID_FILESIZE_BYTES

In [ ]:
# 6) Main runner

def run_downloader():
    ensure_dirs()
    init_log()

    df = pd.read_csv(
    "/content/drive/MyDrive/Dissertation/retracted_dois.csv",
    sep=";"
)
    df = normalize_columns(df)

    if "doi" not in df.columns or "code" not in df.columns:
        raise ValueError("CSV must contain columns named 'doi' and 'code' (case-insensitive).")

    successes, start_idx = load_state()

    existing = 0
    for fn in os.listdir(OUTPUT_DIR):
        if fn.lower().endswith(".pdf"):
            fp = os.path.join(OUTPUT_DIR, fn)
            if os.path.getsize(fp) >= MIN_VALID_FILESIZE_BYTES:
                existing += 1
    if existing > successes:
        successes = existing  # keep progress consistent with disk

    pbar = tqdm(total=TARGET_SUCCESSES, initial=successes, desc="OA PDFs downloaded", unit="pdf")

    for i in range(start_idx, len(df)):
        if successes >= TARGET_SUCCESSES:
            break

        code = str(df.loc[i, "code"])
        doi = str(df.loc[i, "doi"]).strip()

        safe_code = sanitize_filename(code)
        out_file = os.path.join(OUTPUT_DIR, f"{safe_code}.pdf")

        # If exists already, count it
        if already_downloaded(out_file):
            successes += 1
            pbar.update(1)
            save_state(successes, i + 1)
            continue

        row = {
            "index": i,
            "code": code,
            "doi": doi,
            "is_oa": False,
            "pdf_url": "",
            "status": "skipped",
            "message": "",
            "filepath": ""
        }

        try:
            meta = unpaywall_lookup(doi, UNPAYWALL_EMAIL)
            is_oa = bool(meta.get("is_oa")) if meta else False
            pdf_url = extract_oa_pdf_url(meta)

            row["is_oa"] = is_oa
            row["pdf_url"] = pdf_url or ""

            if not is_oa or not pdf_url:
                row["status"] = "skipped"
                row["message"] = "Not OA or no OA PDF URL found"
            else:
                ok, msg = download_pdf(pdf_url, out_file)
                if ok:
                    row["status"] = "downloaded"
                    row["message"] = msg
                    row["filepath"] = out_file
                    successes += 1
                    pbar.update(1)
                else:
                    row["status"] = "failed"
                    row["message"] = msg

        except requests.HTTPError as e:
            row["status"] = "failed"
            row["message"] = f"HTTP error: {e}"
        except requests.RequestException as e:
            row["status"] = "failed"
            row["message"] = f"Request error: {e}"
        except Exception as e:
            row["status"] = "failed"
            row["message"] = f"Unexpected error: {e}"

        append_log(row)
        save_state(successes, i + 1)

        time.sleep(MIN_DELAY_S)

    pbar.close()
    print(f"\nFinished. Successful OA downloads: {successes}/{TARGET_SUCCESSES}")
    print(f"PDF folder: {OUTPUT_DIR}")
    print(f"Log file:   {LOG_PATH}")
    print(f"State file: {STATE_PATH}")

In [ ]:
# 7) Run the downloader

run_downloader()